# Geological Surface Accuracy





# 0.1 Carica il workspace



eseguire (play) la cella "CARICA SPAZIO DI LAVORO" solo all'apertura del notebook (altrimenti si ricarica lo spazio di lavoro di default cancellando le modifiche):



- verrà caricato l'ambiente di lavoro e la cartella "cartella_files" dove andranno messi i file GOCAD delle superfici che costituiscono il modello 3D (.ts) e gli shape file che contengono le tracce delle sezioni sismiche



  e e i punti relativi ai pozzi 





In [ ]:
# ### CARICA SPAZIO DI LAVORO



import os

import shutil



# Percorso base

base_path = '/content'

repo_path = os.path.join(base_path, 'GeoSurface_Accuracy')



# Funzione per pulire completamente la directory

def clean_repo_directory():

    try:

        # Rimuovi la directory se esiste

        if os.path.exists(repo_path):

            shutil.rmtree(repo_path)

            print(f"Directory {repo_path} rimossa")

    except Exception as e:

        print(f"Errore nella rimozione della directory: {e}")



# Pulisci la directory

clean_repo_directory()



# Cambia nella directory base

os.chdir(base_path)



# Clona il repository

!git clone https://github.com/BaterHub/GeoSurface_Accuracy.git



# Cambia nella directory del repository

%cd GeoSurface_Accuracy





# 0.2 Carica i file nella cartella "cartella_files"



Trascinare i file* del pacchetto costituente il modello 3D nella cartella "working_files_folder"



*NB andranno caricati i seguenti file:



- horizons.ts (deve contenere tutte le geometrie delle superfici)



- shapefile delle tracce di sezioni geologiche e linee sismiche utilizzate per la costruzione della superficie





# 0.3 Eseguire lo script



- Posizionarsi nella cella "LANCIA LO SCRIPT" e eseguire il RUN con "ctrl + F10" oppure dal men� "Runtime > Run cell and below/Esegui questa cella e quelle sottostanti"

- Al termine del RUN verranno generati gli output e un log_file all'interno della working_files_folder.





# 1. Importa librerie e funzioni





In [ ]:
# ### LANCIA LO SCRIPT



## Importa librerie necessarie

import pandas as pd

import geopandas as gpd

import numpy as np

import matplotlib.pyplot as plt

from matplotlib.colors import LinearSegmentedColormap

from pyproj import Proj, transform

from scipy.spatial import cKDTree

from scipy.interpolate import griddata

from sklearn.preprocessing import MinMaxScaler

import os

import re

from pathlib import Path



#############################################################################################

## Importa funzioni

import importlib # modulo per il reload delle funzioni



# Reimporta i moduli originali

import files_utils



# Ricarica forzata di ciascun modulo

importlib.reload(files_utils)



# Reimporta le funzioni dai moduli ricaricati

from files_utils import *

#############################################################################################



# Percorso cartella

folder_name = "working_files_folder"

input_dir = os.path.abspath(folder_name)

output_dir = "output_results"





In [ ]:
# ### Impostazione delle cartella di lavoro
working_dir = "working_files_folder"
output_dir = "output_results"
crs = 'EPSG:6708'
grid_spacing = 1000  # distanza nodi griglia (metri)

In [ ]:
# Main Function ed esecuzione procedura

def main(working_dir=working_dir, grid_spacing=grid_spacing):
    print("Avvio dell'analisi dei dati geologici...")

    if not os.path.exists(working_dir):
        print(f"La cartella {working_dir} non esiste. Creazione in corso...")
        os.makedirs(working_dir)
        print(f"Cartella {working_dir} creata. Inserisci i file GOCAD .ts e gli shapefile nella cartella.")
        return None

    print(f"File presenti nella cartella {working_dir}:")
    for file in os.listdir(working_dir):
        print(f"  - {file}")

    ts_files = [f for f in os.listdir(working_dir) if f.endswith('.ts')]
    if not ts_files:
        print("Nessun file .ts trovato.")
        return None
    ts_path = os.path.join(working_dir, ts_files[0])

    surfaces_data = read_gocad_ts_multi(ts_path)
    surface_names = list(surfaces_data.keys())
    if not surface_names:
        print("Nessuna superficie trovata nel file .ts.")
        return None

    wells_all = read_wells_shapefile(working_dir)
    sections_all = read_sections_shapefile(working_dir)

    mapping_flags = {name: ensure_mapping_file(working_dir, name) for name in surface_names}
    edges_df = ensure_checkpoint_edges_file(working_dir, surface_names, wells_all, sections_all)

    results = {}

    for sname, data in surfaces_data.items():
        print(f"\n--- Superficie: {sname} ---")
        vertices = data.get('vertices')
        triangles = data.get('triangles')
        if vertices is None or len(vertices) == 0:
            print("Nessun vertice per questa superficie, salto.")
            continue

        flags = mapping_flags.get(sname, {'use_wells': True, 'use_sections': True, 'use_maps': False})
        wells_use = wells_all if flags.get('use_wells', True) else None
        sections_use = sections_all if flags.get('use_sections', True) else None

        wells_filt, sections_filt = filter_checkpoints_by_edges(edges_df, sname, wells_use, sections_use)

        has_wells = wells_filt is not None and not wells_filt.empty
        has_sections = sections_filt is not None and not sections_filt.empty
        print(f"  Pozzi usati: {'si' if has_wells else 'no'}")
        print(f"  Sezioni usate: {'si' if has_sections else 'no'}")

        acc_outputs = generate_accuracy_outputs(vertices, wells_filt, sections_filt, output_dir,
                                                use_wells=has_wells, use_sections=has_sections,
                                                grid_spacing=grid_spacing, line_step=2000, surface_name=sname)

        try:
            fig = visualize_data(vertices, triangles, wells_filt, sections_filt, apply_smoothing=False,
                                 smoothing_iterations=3, smoothing_factor=0.2, crs='EPSG:6708',
                                 output_filename=f'model_dataset_{sname}.png',
                                 grid_points=acc_outputs.get('grid_points'))
            print("Visualizzazione completata con successo.")
        except Exception as e:
            print(f"Errore durante la visualizzazione: {e}")
            import traceback
            traceback.print_exc()

        results[sname] = {
            'vertices': vertices,
            'triangles': triangles,
            'wells': wells_filt,
            'sections': sections_filt,
            'grid_points': acc_outputs.get('grid_points'),
            'horizontal_weights': acc_outputs.get('weights')
        }

    print("Analisi completata.")
    return results


if __name__ == "__main__":
    data = main()
